# Блок 3 — Ансамблирование и финальная оценка

| Метод | Описание |
|---|---|
| **Hard Voting** | Каждая модель голосует за класс, побеждает большинство |
| **Soft Voting** | Усреднение вероятностей всех моделей |
| **Weighted Averaging** | Усреднение, взвешенное по F1-macro |
| **Stacking (LogReg)** | Мета-модель: логрегрессия на вероятностях |
| **Stacking (SVM)** | Мета-модель: SVM на вероятностях |
| **Stacking (XGBoost)** | Мета-модель: XGBoost на вероятностях |
| **Stacking (GradientBoosting)** | Мета-модель: Gradient Boosting на вероятностях |
| **Temperature Scaling** | Калибровка уверенности каждой модели |

In [ ]:
import sys, os, json

PROJECT_ROOT = '/kaggle/input/datasets/inexyy/se-analysis' if os.path.exists('/kaggle') else os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

WORKING_DIR = '/kaggle/working' if os.path.exists('/kaggle') else '../results'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120

# Загружаем label_names (с fallback на 7 классов Ekman)
_ln_path = f'{WORKING_DIR}/label_names.json'
if os.path.exists(_ln_path):
    with open(_ln_path) as f:
        label_names = json.load(f)
else:
    label_names = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
print(f'Классы: {label_names}')


In [ ]:
# ── Загрузка предсказаний из HuggingFace Hub ─────────────────────────────
# Структура репо: 02_training_*/models/{model_name}/*.npy
# Stage-1 папки (*_s1) пропускаются — берём только финальные Stage-2.
# Скачивает только .npy + results.json (без весов моделей, ~несколько МБ).
import shutil
from huggingface_hub import snapshot_download

HF_MODELS_REPO = 'Kirillx/ru-text-emotion-classification'
MODELS_BASE     = os.path.join(WORKING_DIR, 'models')

def _preds_ready(base, n=2):
    if not os.path.isdir(base):
        return False
    return sum(
        os.path.exists(os.path.join(base, d, 'test_probs.npy'))
        for d in os.listdir(base)
        if os.path.isdir(os.path.join(base, d))
    ) >= n

if _preds_ready(MODELS_BASE):
    print(f'Предсказания уже на месте: {MODELS_BASE}')
else:
    print(f'Скачиваем из {HF_MODELS_REPO} …')
    hf_dir = snapshot_download(
        repo_id=HF_MODELS_REPO,
        local_dir='/tmp/hf_models',
        allow_patterns=['**/*.npy', '**/results.json', '**/label_names.json'],
    )

    # Динамически находим все папки с test_probs.npy, пропуская Stage-1 (_s1)
    os.makedirs(MODELS_BASE, exist_ok=True)
    found = {}
    for root, dirs, files in os.walk(hf_dir):
        if 'test_probs.npy' in files:
            key = os.path.basename(root)
            if not key.endswith('_s1'):
                found[key] = root

    print(f'Найдено {len(found)} модел(ей): {sorted(found)}')
    for key, src in sorted(found.items()):
        dst = os.path.join(MODELS_BASE, key)
        os.makedirs(dst, exist_ok=True)
        for fname in os.listdir(src):
            if fname.endswith('.npy') or fname == 'results.json':
                shutil.copy2(os.path.join(src, fname), os.path.join(dst, fname))
        print(f'  {key} ✓')

    # label_names.json → WORKING_DIR (берём первый найденный)
    ln_dst = os.path.join(WORKING_DIR, 'label_names.json')
    if not os.path.exists(ln_dst):
        for root, dirs, files in os.walk(hf_dir):
            if 'label_names.json' in files:
                shutil.copy2(os.path.join(root, 'label_names.json'), ln_dst)
                with open(ln_dst) as f:
                    label_names = json.load(f)
                print(f'label_names: {label_names}')
                break

    print('Готово.')


## 1. Загрузка предсказаний

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

MODEL_DIRS = {
    'ruBERT':            f'{WORKING_DIR}/models/rubert',
    'XLM-RoBERTa':       f'{WORKING_DIR}/models/xlmroberta',
    'ruBERT-tiny':       f'{WORKING_DIR}/models/rubert_tiny',
    'ruBERT-large':      f'{WORKING_DIR}/models/rubert_large',
    'ruRoBERTa-large':   f'{WORKING_DIR}/models/ruroberta_large',
    'Aniemore-emotion':  f'{WORKING_DIR}/models/aniemore_emotion',
    'seara-goem':        f'{WORKING_DIR}/models/seara_goem',
}

all_probs  = {}
all_preds  = {}
all_val_f1 = {}
all_labels = None

for name, model_dir in MODEL_DIRS.items():
    probs_path  = os.path.join(model_dir, 'test_probs.npy')
    preds_path  = os.path.join(model_dir, 'test_preds.npy')
    labels_path = os.path.join(model_dir, 'test_labels.npy')

    if not os.path.exists(probs_path):
        print(f'WARNING: {name} — предсказания не найдены в {model_dir}')
        continue

    all_probs[name] = np.load(probs_path)
    all_preds[name] = np.load(preds_path)
    labels = np.load(labels_path)
    if all_labels is None:
        all_labels = labels

    results_path = os.path.join(model_dir, 'results.json')
    if os.path.exists(results_path):
        with open(results_path) as f:
            res = json.load(f)
        all_val_f1[name] = res.get('test_report', {}).get('macro avg', {}).get('f1-score', 0.0)
    else:
        all_val_f1[name] = f1_score(all_labels, all_preds[name], average='macro')

    print(f'{name:<22s}: {len(all_preds[name]):,} примеров | F1-macro={all_val_f1[name]:.4f}')

if len(all_probs) < 2:
    raise RuntimeError('Нужно как минимум 2 обученных модели.')

In [ ]:
from src.evaluation import evaluate_predictions, compare_models

individual_results = []
for name in all_probs:
    m = evaluate_predictions(
        all_labels, all_preds[name], all_probs[name],
        model_name=name, label_names=label_names,
    )
    individual_results.append(m)

df_ind = pd.DataFrame(individual_results).set_index('model')
print(df_ind[['accuracy', 'f1_macro', 'f1_weighted']].round(4).to_string())


## 2. Ансамбли

In [ ]:
from src.ensemble import (
    hard_voting, soft_voting, weighted_averaging,
    stacking_ensemble, evaluate_ensemble,
)

probs_list = list(all_probs.values())
preds_list = list(all_preds.values())
f1_weights = list(all_val_f1.values())

# ── Hard Voting
hv = hard_voting(preds_list)
print(f'Hard Voting   — F1-macro: {f1_score(all_labels, hv, average="macro"):.4f}')

# ── Soft Voting
sv = soft_voting(probs_list)
print(f'Soft Voting   — F1-macro: {f1_score(all_labels, sv, average="macro"):.4f}')

# ── Weighted Averaging
wa = weighted_averaging(probs_list, f1_weights)
print(f'Weighted Avg  — F1-macro: {f1_score(all_labels, wa, average="macro"):.4f}')


In [ ]:
# ── Stacking
# Для мета-обучения берём val-предсказания (без утечки данных)
val_probs_list = []
val_labels_arr = None

for name, model_dir in MODEL_DIRS.items():
    if name not in all_probs:
        continue
    vpath = os.path.join(model_dir, 'val_probs.npy')
    if os.path.exists(vpath):
        val_probs_list.append(np.load(vpath))
        if val_labels_arr is None:
            val_labels_arr = np.load(os.path.join(model_dir, 'val_labels.npy'))
    else:
        n = len(all_probs[name]) // 2
        val_probs_list.append(all_probs[name][:n])
        if val_labels_arr is None:
            val_labels_arr = all_labels[:n]

n_half = len(all_labels) // 2
test_probs_half  = [p[n_half:] for p in list(all_probs.values())]
test_labels_half = all_labels[n_half:]

print('Stacking — Logistic Regression:')
st_lr, _ = stacking_ensemble(val_probs_list, val_labels_arr, test_probs_half, meta_learner='logistic')
print(f'  F1-macro: {f1_score(test_labels_half, st_lr, average="macro"):.4f}')

print('\nStacking — SVM:')
st_svm, _ = stacking_ensemble(val_probs_list, val_labels_arr, test_probs_half, meta_learner='svm')
print(f'  F1-macro: {f1_score(test_labels_half, st_svm, average="macro"):.4f}')

print('\nStacking — XGBoost:')
st_xgb, _ = stacking_ensemble(val_probs_list, val_labels_arr, test_probs_half, meta_learner='xgboost')
print(f'  F1-macro: {f1_score(test_labels_half, st_xgb, average="macro"):.4f}')

print('\nStacking — Gradient Boosting:')
st_gb, _ = stacking_ensemble(val_probs_list, val_labels_arr, test_probs_half, meta_learner='gradient_boosting')
print(f'  F1-macro: {f1_score(test_labels_half, st_gb, average="macro"):.4f}')

## 2b. Калибровка температуры (Temperature Scaling)

In [ ]:
from src.ensemble import fit_temperature, temperature_scaling
from sklearn.metrics import log_loss

print('Подбор температуры на val-предсказаниях:')
calibrated_probs = {}
temperatures = {}

for name, model_dir in MODEL_DIRS.items():
    if name not in all_probs:
        continue
    val_probs_path  = os.path.join(model_dir, 'val_probs.npy')
    val_labels_path = os.path.join(model_dir, 'val_labels.npy')
    if not os.path.exists(val_probs_path):
        print(f'  {name}: val_probs.npy не найден, пропускаем')
        continue

    vp = np.load(val_probs_path)
    vl = np.load(val_labels_path).astype(int)

    # Восстанавливаем логиты из softmax-вероятностей (обратное через log)
    # Примечание: fit_temperature принимает логиты — берём log(prob) как приближение
    import scipy.special
    logits_approx = np.log(np.clip(vp, 1e-9, 1.0))

    T = fit_temperature(logits_approx, vl)
    temperatures[name] = T
    cal_probs = temperature_scaling(logits_approx, T)
    calibrated_probs[name] = cal_probs

    nll_before = log_loss(vl, vp)
    nll_after  = log_loss(vl, cal_probs)
    print(f'  {name:<20s}: T={T:.3f} | NLL before={nll_before:.4f} → after={nll_after:.4f}')

print('\nКалиброванный soft voting:')
if calibrated_probs:
    from src.ensemble import soft_voting_proba
    cal_list = [calibrated_probs[n] for n in all_probs if n in calibrated_probs]
    test_cal_list = [all_probs[n] for n in all_probs if n in calibrated_probs]
    sv_cal = np.argmax(soft_voting_proba(test_cal_list), axis=-1)
    from sklearn.metrics import f1_score as _f1
    print(f'  F1-macro: {_f1(all_labels, sv_cal, average="macro"):.4f}')


## 3. Сравнение всех методов

In [ ]:
ensemble_dict = {
    'Hard Voting':        hv,
    'Soft Voting':        sv,
    'Weighted Averaging': wa,
}
df_ens = evaluate_ensemble(all_labels, ensemble_dict)

df_stk = evaluate_ensemble(
    test_labels_half,
    {
        'Stacking LogReg':    st_lr,
        'Stacking SVM':       st_svm,
        'Stacking XGBoost':   st_xgb,
        'Stacking GradBoost': st_gb,
    },
)

print('=== Ансамбли (полная тест. выборка) ===')
print(df_ens.round(4).to_string())
print('\n=== Стекинг (половина тест. выборки) ===')
print(df_stk.round(4).to_string())


In [ ]:
best_name = df_ens['f1_macro'].idxmax()
best_preds = ensemble_dict[best_name]
best_metrics = evaluate_predictions(
    all_labels, best_preds, model_name=f'Ensemble ({best_name})', label_names=label_names
)
all_rows = individual_results + [best_metrics]
compare_models(all_rows, save_path=f'{WORKING_DIR}/model_comparison.png')
print(f'\nЛучший ансамбль: {best_name} | F1-macro: {best_metrics["f1_macro"]:.4f}')


In [ ]:
from src.evaluation import plot_confusion_matrix
plot_confusion_matrix(
    all_labels, best_preds,
    model_name=f'Best: {best_name}',
    label_names=label_names,
    save_path=f'{WORKING_DIR}/cm_best_ensemble.png',
)


## 4. Итоговый отчёт

In [ ]:
from sklearn.metrics import classification_report

print('\n' + '='*65)
print('ИТОГОВЫЙ ОТЧЁТ — Классификация эмоций (Ekman, 7 классов)')
print('='*65)
print('\n── Индивидуальные модели ──')
print(df_ind[['accuracy', 'f1_macro', 'f1_weighted']].round(4).to_string())
print('\n── Ансамбли (полная тест. выборка) ──')
print(df_ens.round(4).to_string())
print('\n── Стекинг (половина тест. выборки) ──')
print(df_stk.round(4).to_string())
print(f'\n── Лучший ансамбль: {best_name} ──')
print(classification_report(all_labels, best_preds, target_names=label_names))

summary = {
    'task': 'emotion classification (Ekman 7-class)',
    'label_names': label_names,
    'individual_models': df_ind.round(4).to_dict(),
    'ensemble_methods': df_ens.round(4).to_dict(),
    'stacking_methods': df_stk.round(4).to_dict(),
    'best_ensemble': best_name,
    'best_f1_macro': round(float(best_metrics['f1_macro']), 4),
}
with open(f'{WORKING_DIR}/final_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print(f'\nSummary сохранён: {WORKING_DIR}/final_summary.json')


## 5. Сохранение финальной модели-ансамбля

In [ ]:
from src.ensemble import save_ensemble

ENSEMBLE_DIR = os.path.join(WORKING_DIR, 'ensemble')

# Best across voting methods (full test set)
best_voting_name = df_ens['f1_macro'].idxmax()
best_voting_f1   = float(df_ens.loc[best_voting_name, 'f1_macro'])

# Best across stacking methods (half test set — comparable within stacking)
best_stk_name = df_stk['f1_macro'].idxmax()
best_stk_f1   = float(df_stk.loc[best_stk_name, 'f1_macro'])

print(f'Лучший voting   : {best_voting_name:<22s} F1={best_voting_f1:.4f}')
print(f'Лучший stacking : {best_stk_name:<22s} F1={best_stk_f1:.4f}')

_meta_learner_map = {
    'Stacking LogReg':    'logistic',
    'Stacking SVM':       'svm',
    'Stacking XGBoost':   'xgboost',
    'Stacking GradBoost': 'gradient_boosting',
}

if best_voting_f1 >= best_stk_f1:
    final_method   = best_voting_name
    final_f1       = best_voting_f1
    final_weights  = f1_weights if best_voting_name == 'Weighted Averaging' else None
    final_meta_clf = None
    print(f'\n→ Победитель: voting [{final_method}]')
else:
    final_method  = best_stk_name
    final_f1      = best_stk_f1
    final_weights = None
    # Re-fit meta-learner on the full validation set for the saved model
    print(f'\n→ Победитель: stacking [{final_method}]')
    print('  Переобучаем мета-модель на полном val-наборе...')
    _, final_meta_clf = stacking_ensemble(
        val_probs_list, val_labels_arr,
        list(all_probs.values()),
        meta_learner=_meta_learner_map[final_method],
    )

model_dirs_list = [MODEL_DIRS[n] for n in all_probs]

save_ensemble(
    save_dir    = ENSEMBLE_DIR,
    method      = final_method,
    model_dirs  = model_dirs_list,
    weights     = final_weights,
    meta_clf    = final_meta_clf,
    label_names = label_names,
    metrics     = {'f1_macro': round(final_f1, 4)},
)

print(f'\n✓ Сохранено: {ENSEMBLE_DIR}')
for fn in sorted(os.listdir(ENSEMBLE_DIR)):
    size = os.path.getsize(os.path.join(ENSEMBLE_DIR, fn))
    print(f'   {fn}  ({size:,} bytes)')

print(f'\nИспользование в инференсе:')
print(f'  from src.inference import EmotionClassifier')
print(f'  clf = EmotionClassifier.from_config("{ENSEMBLE_DIR}")')
print(f'  clf.predict_label(["Мне очень страшно идти туда одному"])')

## 6. Дистилляция: от 3 учителей к одной модели

Объединяем знания трёх лучших загруженных моделей в один `xlm-roberta-base` через **Knowledge Distillation**.

```
ruroberta_large ─┐
xlmroberta       ├─→ avg(probs) = мягкие метки   →  xlm-roberta-base (студент)
seara_goem      ─┘                                    Loss = α·T²·KL + (1−α)·CE
```

| T (температура) | α (вес KL) | LR | Epochs |
|---|---|---|---|
| 2.0 | 0.7 | 3e-5 | 5 |

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from datasets import load_from_disk
from src.preprocessor import clean_text
import time

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── Конфигурация дистилляции ──────────────────────────────────────────────
TEACHER_KEYS    = ['ruroberta_large', 'xlmroberta', 'seara_goem']
STUDENT_BASE_ID = 'xlm-roberta-base'
STUDENT_DIR     = os.path.join(WORKING_DIR, 'models', 'distilled_xlmr')
SOFT_LABELS_PATH = os.path.join(WORKING_DIR, 'distill_soft_labels_train.npy')

T_DISTILL  = 2.0    # температура: размягчает распределения
ALPHA      = 0.7    # вес KL-потери (мягкие метки)
LR         = 3e-5
EPOCHS     = 5
BATCH_SIZE = 32
MAX_LENGTH = 128
N_CLASSES  = len(label_names)

print(f'Device: {DEVICE} | Студент: {STUDENT_BASE_ID}')
print(f'Учителя: {TEACHER_KEYS}')
print(f'T={T_DISTILL}, α={ALPHA}, LR={LR}, epochs={EPOCHS}')


In [ ]:
# ── Загрузка Stage-2 train ────────────────────────────────────────────────
for cand in [
    os.path.join(WORKING_DIR, 'stage2_data_augmented'),
    os.path.join(WORKING_DIR, 'stage2_data'),
    os.path.join(PROJECT_ROOT, 'data', 'stage2_data_augmented'),
    os.path.join(PROJECT_ROOT, 'data', 'stage2_data'),
]:
    if os.path.exists(cand):
        s2_ds = load_from_disk(cand)
        print(f'Stage-2: {cand}')
        break

train_texts  = [clean_text(t) for t in s2_ds['train']['text']]
train_labels_arr = np.array(s2_ds['train']['label'])
test_texts_d = [clean_text(t) for t in s2_ds['test']['text']]
test_labels_d = np.array(s2_ds['test']['label'])
print(f'Train: {len(train_texts):,} | Test: {len(test_texts_d):,}')

# ── Генерация мягких меток от учителей (с кешем) ─────────────────────────
def _teacher_probs(model_dir, texts, batch_size=64):
    tok   = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(DEVICE).eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            enc = tok(texts[i:i+batch_size], truncation=True, max_length=MAX_LENGTH,
                      padding=True, return_tensors='pt').to(DEVICE)
            out.append(torch.softmax(model(**enc).logits, -1).cpu().numpy())
    del model; torch.cuda.empty_cache()
    return np.concatenate(out)

if os.path.exists(SOFT_LABELS_PATH):
    soft_labels_train = np.load(SOFT_LABELS_PATH)
    print(f'Мягкие метки загружены из кеша: {soft_labels_train.shape}')
else:
    teacher_list = []
    for key in TEACHER_KEYS:
        tdir = os.path.join(WORKING_DIR, 'models', key)
        if not os.path.exists(tdir):
            print(f'  WARN: {key} не найден — пропускаем')
            continue
        print(f'  Инференс учителя {key} ({len(train_texts):,} примеров) ...')
        teacher_list.append(_teacher_probs(tdir, train_texts))
        print(f'    ✓ mean_max={teacher_list[-1].max(1).mean():.3f}')
    soft_labels_train = np.mean(teacher_list, axis=0)
    np.save(SOFT_LABELS_PATH, soft_labels_train)
    print(f'Мягкие метки сохранены: {soft_labels_train.shape}')


In [ ]:
# ── Dataset + функция потерь ──────────────────────────────────────────────
class DistillDataset(Dataset):
    def __init__(self, texts, hard_labels, soft_labels, tokenizer):
        self.enc = tokenizer(texts, truncation=True, max_length=MAX_LENGTH,
                             padding='max_length', return_tensors='pt')
        self.hard = torch.tensor(hard_labels, dtype=torch.long)
        self.soft = torch.tensor(soft_labels, dtype=torch.float32)

    def __len__(self): return len(self.hard)

    def __getitem__(self, i):
        return {
            'input_ids':      self.enc['input_ids'][i],
            'attention_mask': self.enc['attention_mask'][i],
            'hard_label':     self.hard[i],
            'soft_label':     self.soft[i],
        }

def distill_loss(logits, hard, soft, T=T_DISTILL, alpha=ALPHA):
    ce  = F.cross_entropy(logits, hard)
    log_p_s = F.log_softmax(logits / T, dim=-1)
    p_t_T   = F.softmax(torch.log(soft.clamp(1e-10)) / T, dim=-1)
    kl  = F.kl_div(log_p_s, p_t_T, reduction='batchmean') * T ** 2
    return alpha * kl + (1 - alpha) * ce, kl.item(), ce.item()

@torch.no_grad()
def eval_model(model, texts, labels, batch_size=64):
    tok = student_tok
    model.eval()
    preds = []
    for i in range(0, len(texts), batch_size):
        enc = tok(texts[i:i+batch_size], truncation=True, max_length=MAX_LENGTH,
                  padding=True, return_tensors='pt').to(DEVICE)
        preds.append(model(**enc).logits.argmax(-1).cpu().numpy())
    preds = np.concatenate(preds)
    return accuracy_score(labels, preds), f1_score(labels, preds, average='macro', zero_division=0), preds

# ── Инициализация студента ────────────────────────────────────────────────
student_tok = AutoTokenizer.from_pretrained(STUDENT_BASE_ID)
student = AutoModelForSequenceClassification.from_pretrained(
    STUDENT_BASE_ID,
    num_labels=N_CLASSES,
    id2label={i: n for i, n in enumerate(label_names)},
    label2id={n: i for i, n in enumerate(label_names)},
).to(DEVICE)

train_ds     = DistillDataset(train_texts, train_labels_arr, soft_labels_train, student_tok)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

total_steps  = len(train_loader) * EPOCHS
optimizer    = AdamW(student.parameters(), lr=LR, weight_decay=0.01)
scheduler    = get_linear_schedule_with_warmup(optimizer, total_steps // 10, total_steps)

print(f'Параметры студента: {sum(p.numel() for p in student.parameters()):,}')
print(f'Шагов обучения: {total_steps} ({len(train_loader)} батчей × {EPOCHS} эпох)')


In [ ]:
# ── Цикл обучения ─────────────────────────────────────────────────────────
best_f1_d, best_epoch_d, history_d = 0.0, 0, []

for epoch in range(1, EPOCHS + 1):
    student.train()
    t_loss = t_kl = t_ce = 0.0
    t0 = time.time()

    for batch in train_loader:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        hard = batch['hard_label'].to(DEVICE)
        soft = batch['soft_label'].to(DEVICE)

        logits = student(input_ids=ids, attention_mask=mask).logits
        loss, kl, ce = distill_loss(logits, hard, soft)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        t_loss += loss.item(); t_kl += kl; t_ce += ce

    n = len(train_loader)
    acc, f1, _ = eval_model(student, test_texts_d, test_labels_d)
    elapsed = time.time() - t0
    print(f'Epoch {epoch}/{EPOCHS} | loss={t_loss/n:.4f} (KL={t_kl/n:.4f} CE={t_ce/n:.4f}) | '
          f'acc={acc:.4f} F1={f1:.4f} | {elapsed:.0f}s')
    history_d.append({'epoch': epoch, 'loss': t_loss/n, 'kl': t_kl/n, 'ce': t_ce/n,
                      'acc': acc, 'f1_macro': f1})

    if f1 > best_f1_d:
        best_f1_d, best_epoch_d = f1, epoch
        os.makedirs(STUDENT_DIR, exist_ok=True)
        student.save_pretrained(STUDENT_DIR)
        student_tok.save_pretrained(STUDENT_DIR)
        print(f'  ✓ Лучший чекпоинт сохранён (F1={best_f1_d:.4f})')

print(f'\nОбучение завершено | Лучший epoch: {best_epoch_d} | F1-macro: {best_f1_d:.4f}')


In [ ]:
from sklearn.metrics import classification_report as cr

# ── Финальная оценка студента ──────────────────────────────────────────────
student_best = AutoModelForSequenceClassification.from_pretrained(STUDENT_DIR).to(DEVICE).eval()
acc_d, f1_d, preds_d = eval_model(student_best, test_texts_d, test_labels_d)

rows_d = [{'model': 'distilled_xlmr (студент)', 'accuracy': acc_d, 'f1_macro': f1_d}]
for key in TEACHER_KEYS:
    pp = os.path.join(WORKING_DIR, 'models', key, 'test_preds.npy')
    lp = os.path.join(WORKING_DIR, 'models', key, 'test_labels.npy')
    if os.path.exists(pp) and os.path.exists(lp):
        tp, tl = np.load(pp), np.load(lp)
        rows_d.append({'model': key,
                       'accuracy': accuracy_score(tl, tp),
                       'f1_macro': f1_score(tl, tp, average='macro', zero_division=0)})

df_distill = pd.DataFrame(rows_d).set_index('model').sort_values('f1_macro', ascending=False)
print('=== Студент vs Учителя ===')
print(df_distill.round(4).to_string())
print(f'\n=== Отчёт по классам (дистиллированная модель) ===')
print(cr(test_labels_d, preds_d, target_names=label_names))

# ── Кривые обучения ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ep = [h['epoch'] for h in history_d]
axes[0].plot(ep, [h['loss'] for h in history_d], 'b-o', label='Total')
axes[0].plot(ep, [h['kl']   for h in history_d], 'g--', label='KL')
axes[0].plot(ep, [h['ce']   for h in history_d], 'r--', label='CE')
axes[0].set_title('Потери дистилляции'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(ep, [h['f1_macro'] for h in history_d], 'b-o')
axes[1].axhline(best_f1_d, color='green', linestyle='--', label=f'Best F1={best_f1_d:.4f}')
axes[1].set_title('F1-macro на тесте'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{WORKING_DIR}/distillation_training.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Сохранение результатов ────────────────────────────────────────────────
with open(f'{WORKING_DIR}/distillation_results.json', 'w', encoding='utf-8') as f:
    json.dump({
        'student': STUDENT_BASE_ID, 'teachers': TEACHER_KEYS,
        'hyperparams': {'T': T_DISTILL, 'alpha': ALPHA, 'lr': LR, 'epochs': EPOCHS},
        'best_epoch': best_epoch_d, 'test_f1_macro': round(f1_d, 4),
        'test_accuracy': round(acc_d, 4), 'history': history_d,
    }, f, ensure_ascii=False, indent=2)

print(f'\n✓ Модель: {STUDENT_DIR}')
print(f'  F1-macro: {f1_d:.4f} | Accuracy: {acc_d:.4f}')
print(f'\nИнференс:')
print(f'  from src.inference import EmotionClassifier')
print(f'  clf = EmotionClassifier("{STUDENT_DIR}")')
